In [ ]:
import streamlit as st
import requests
import pandas as pd
import plotly.graph_objects as go
import numpy as np
from scipy import stats as scipy_stats
from datetime import date

API_URL = "http://127.0.0.1:8000/global_predict"

st.set_page_config(layout="wide")
st.title("Bitcoin — Signal ML")

# ── SIDE PANEL ──────────────────────────────────────────────────
with st.sidebar:
    st.header("Paramètres")

    date_pivot = st.date_input("Date cutoff training", value=date(2024, 1, 1), min_value=date(2023, 11, 4), max_value=date.today())
    date_pivot = date_pivot.strftime("%Y-%m-%d")

    st.divider()
    st.subheader("Stratégies")

    show_signal = st.checkbox("Signal ML (threshold 0.5)", value=True)
    show_flex = st.checkbox("Signal personnalisé (threshold ajustable)", value=True)
    show_long_only = st.checkbox("Long only", value=True)
    show_random = st.checkbox("Random", value=False)

    st.divider()
    st.subheader("Seuil de proba pour signal d'achat")
    threshold = st.slider("Seuil", min_value=0.5, max_value=0.75, value=0.5, step=0.005)

    horizon = 5

    st.divider()
    st.subheader("Modèle")
    model_name = st.selectbox(
    "Modèle",
    options=["xgb", "rnn", "linear"],
    format_func=lambda x: {"xgb": "XGBoost", "rnn": "RNN", "linear": "Régression linéaire"}[x]
    )

    load = st.button("Charger les données", type="primary")


2026-04-22 20:49:41.977 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 20:49:41.979 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 20:49:42.819 
  command:

    streamlit run /home/geoff/.pyenv/versions/ai_for_finance/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-04-22 20:49:42.820 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 20:49:42.821 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 20:49:42.823 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 20:49:42.824 Thread 'MainThread': missing ScriptRunContext! This war

In [4]:
# ── DATA LOADING ─────────────────────────────────────────────────

response = requests.get(API_URL, params={"date_pivot": date_pivot, "model_name": 'linear'})
data = response.json()


In [6]:
data["df_for_streamlit"]

[{'Date': '2024-01-01T00:00:00',
  'Open': 42280.234375,
  'High': 44175.4375,
  'Low': 42214.9765625,
  'Close': 44167.33203125,
  'Volume': 18426978443,
  'prediction': 1,
  'probability': 0.5491571486725167},
 {'Date': '2024-01-02T00:00:00',
  'Open': 44187.140625,
  'High': 45899.70703125,
  'Low': 44176.94921875,
  'Close': 44957.96875,
  'Volume': 39335274536,
  'prediction': 1,
  'probability': 0.5491571486725167},
 {'Date': '2024-01-03T00:00:00',
  'Open': 44961.6015625,
  'High': 45503.2421875,
  'Low': 40813.53515625,
  'Close': 42848.17578125,
  'Volume': 46342323118,
  'prediction': 1,
  'probability': 0.5491571486725167},
 {'Date': '2024-01-04T00:00:00',
  'Open': 42855.81640625,
  'High': 44770.0234375,
  'Low': 42675.17578125,
  'Close': 44179.921875,
  'Volume': 30448091210,
  'prediction': 1,
  'probability': 0.5491571486725167},
 {'Date': '2024-01-05T00:00:00',
  'Open': 44192.98046875,
  'High': 44353.28515625,
  'Low': 42784.71875,
  'Close': 44162.69140625,
  'Volu

In [ ]:
df = pd.DataFrame(data["df_for_streamlit"])
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)
st.session_state["df"] = df